# Synthetic geometry audit: how much of the position error is our algorithm and how much is noise in the data? (iteration 40)

**Why.** Below 800 m about 35-40 % of the answers on the new set miss the 1 sigma band (median error 0.66-0.89 sigma), against 15 % on the old official set and 1 % on NUDT. Refitting the final position gave only +0.67 of 60 (iteration 40). Codex (2026-11-18) proposed isolating geometry errors with satellite-derived photos of known camera position.

**Method.** For every development photo below 1000 m (202), a synthetic 320 x 256 photo is rendered from the map store's own level-3 picture with the real camera (intrinsics, distortion), the real pose and the pipeline's own geometry conventions (`settings.GEOMETRY`: attitude signs, yaw offset, height scale), by projecting each photo pixel to the ground and sampling the map there. The truth is the real photo's GPS truth. The pipeline then runs on these photos exactly as on real ones (hint jittered 5 km, 6 queries, frozen configuration).
- Arm A, **exact metadata**: the pipeline sees the pose the picture was rendered with. Any error is the algorithm's (query quantisation in heading and scale, rounding, the matcher fit, the final shift), with a perfect picture and a perfect map.
- Arm B, **noisy metadata**: pitch and roll off by N(0, 1 degree), yaw by N(0, 2 degrees), height by N(0, 2 %), as a real IMU and altimeter might be. Any error beyond arm A is what metadata noise costs.
Compare the error in sigma of both arms with the real photos of the same height bands.

**Reading rule (written before the run).** If arm A gives a median error of 0.3 sigma or less and at least 90 % within 1 sigma, the algorithm is not the limit and the real-photo error (median 0.66-0.89 sigma below 800 m) comes from the data (metadata noise, GPS truth, map registration, thermal-to-visible differences), which no refit can remove. If arm B reproduces the real error, metadata noise is enough to explain it and pose estimation from the matches (which does not use the noisy attitude) could recover it. If arm A is already bad, there is a convention or geometry error to fix.

Env: `SG_HEADLESS, SG_STORE, SG_META, SG_OUT, SG_LIMIT`.

In [ ]:
import os, sys, math
import numpy as np, pandas as pd, cv2
HEADLESS = os.environ.get("SG_HEADLESS", "../../../../Main_Pipeline_Headless"); sys.path.insert(0, os.path.abspath(HEADLESS))
import settings
settings.LEVELS = [3, 4, 5, 6, 7, 8, 9, 10, 11]
from geoloc import photo
from geoloc.map_store import MapStore
STORE = os.environ["SG_STORE"]; META = os.environ["SG_META"]; OUT = os.environ["SG_OUT"]; LIMIT = int(os.environ.get("SG_LIMIT", 0))
store = MapStore(STORE); meta = pd.read_csv(META); meta = meta[meta.alt < 1000].reset_index(drop=True)
if LIMIT: meta = meta.iloc[:LIMIT]
for arm in ("exact", "noisy"): os.makedirs(os.path.join(OUT, arm, "images"), exist_ok=True)
level_image = store.levels[3]; gsd3 = 0.3 * 2 ** (3 / 2); print(len(meta), "photos below 1000 m; level 3 picture", level_image.shape, "pixel %.3f m" % gsd3)

In [ ]:
def render(row):
    camera = photo.read_camera(row); pose = photo.read_pose(row); g = settings.GEOMETRY
    pitch = roll = 0.0
    if g["attitude"]: pitch = g["pitch_sign"] * pose["pitch"]; roll = g["roll_sign"] * pose["roll"]
    ax, ay, az = photo.camera_axes(pose["yaw"] + g["yaw_offset"], pitch, roll)
    height = (pose["height"] + g["height_offset"]) * g["scale"]
    intrinsic = np.array([[camera["fx"], 0, camera["cx"]], [0, camera["fy"], camera["cy"]], [0, 0, 1]], float)
    dist = np.array([camera["k1"], camera["k2"], camera["p1"], camera["p2"], camera["k3"]])
    cols, rows = np.meshgrid(np.arange(320), np.arange(256)); pts = np.stack([cols, rows], -1).reshape(-1, 1, 2).astype(np.float64)
    und = cv2.undistortPointsIter(pts, intrinsic, dist, None, None, (cv2.TERM_CRITERIA_COUNT + cv2.TERM_CRITERIA_EPS, 50, 1e-10)).reshape(-1, 2)
    ray = und[:, :1] * ax + und[:, 1:] * ay + az; distance = -height / ray[:, 2]
    east = (ray[:, 0] * distance).reshape(256, 320); north = (ray[:, 1] * distance).reshape(256, 320)
    x_m, y_m = store.gps_to_metres(float(row["lat_gtruth"]), float(row["lon_gtruth"]))
    map_x = ((x_m + east) / gsd3 - 0.5).astype(np.float32); map_y = ((y_m - north) / gsd3 - 0.5).astype(np.float32)
    x0 = int(max(map_x.min() - 4, 0)); x1 = int(min(map_x.max() + 5, level_image.shape[1])); y0 = int(max(map_y.min() - 4, 0)); y1 = int(min(map_y.max() + 5, level_image.shape[0]))
    window = np.asarray(level_image[y0:y1, x0:x1])
    return cv2.remap(window, map_x - x0, map_y - y0, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
rng = np.random.default_rng(0); noisy_rows = []; exact_rows = []
for i, row in meta.iterrows():
    image = render(row)
    name = "syn_%04d.png" % i
    for arm in ("exact", "noisy"): cv2.imwrite(os.path.join(OUT, arm, "images", name), image.astype(np.uint16) * 257)
    exact = row.copy(); exact["filename"] = name; exact["lat_sc"] = row["lat_gtruth"]; exact["lon_sc"] = row["lon_gtruth"]; exact_rows.append(exact)
    noisy = exact.copy(); noisy["pitch"] = float(row["pitch"]) + rng.normal(0, 1.0); noisy["roll"] = float(row["roll"]) + rng.normal(0, 1.0)
    noisy["yaw"] = float(row["yaw"]) + rng.normal(0, 2.0); noisy["alt"] = float(row["alt"]) * (1 + rng.normal(0, 0.02)); noisy_rows.append(noisy)
    if i % 50 == 0: print(i, flush=True)
pd.DataFrame(exact_rows).to_csv(os.path.join(OUT, "exact", "metadata.csv"), index=False); pd.DataFrame(noisy_rows).to_csv(os.path.join(OUT, "noisy", "metadata.csv"), index=False)
print("synthetic photos and metadata written to", OUT)